# Pipeline conversacional de Chatbot com NLP (chatbot)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, spaCy, Scikit-Learn  
- **Arquivo Original:** `Chatbot.ipynb`  

---


In [1]:
import numpy as np

In [2]:
# ou, para o grande:
!pip install -q pt-core-news-lg
!python -m spacy download pt_core_news_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 568.2/568.2 MB 1.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
! pip install gensim

In [4]:
import numpy as np
import spacy
import re
import gensim
from gensim.models import Word2Vec

## Carregamento e pré processamento

In [17]:

# carrega modelo PT
nlp = spacy.load("pt_core_news_lg")

# lê o livro
with open("o-guarani.txt", "r", encoding="utf-8") as f:
    raw = f.read()

# limpeza básica
raw = re.sub(r"\s+", " ", raw)               # quita quebras extras
#raw = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ0-9 ]+", "", raw)  # só letras/números

# aplica SpaCy: tokenização + lematização + remoção de stopwords/pontuação
docs = nlp(raw.lower())
tokens = [tok.text for tok in docs]

print(f"Total de tokens limpos: {len(tokens)}")
# exibe primeiras 20
print(tokens[:20])


Total de tokens limpos: 124102
[' ', 'primeira', 'parte', 'os', 'aventureiros', 'i', 'cenário', 'de', 'um', 'dos', 'cabeços', 'da', 'serra', 'dos', 'órgãos', 'desliza', 'um', 'fio', 'de', 'água']


Word2vec

In [18]:


sentences = [[tok.lemma_ for tok in sent if not tok.is_stop and not tok.is_punct]
             for sent in nlp(raw).sents]

# configura e treina
w2v = Word2Vec(
    sentences,
    vector_size=100,    # dimensão do embedding
    window=5,           # contexto
    min_count=2,
    sg=1,               # Skip‑gram
    epochs=10,
    workers=2
)

# salva modelo
w2v.save("w2v_o_guarani.model")
print("Word2Vec treinado e salvo.")


Word2Vec treinado e salvo.


## Chunks

In [19]:
# converte tokens em lista
tok_list = tokens
chunk_size = 250
step = chunk_size // 2

chunk_texts = []  # textos originais para exibir na resposta
chunks = []       # lista de tokens lematizados para vetores

for start in range(0, len(tokens), step):
    end = start + chunk_size
    chunk = tokens[start:end]
    if len(chunk) < 50:
        break
    chunks.append(chunk)
    # cria string original com os mesmos índices de `tokens`
    chunk_text = " ".join(chunk)  # se quiser só mostrar como texto simples
    chunk_texts.append(chunk_text)


print(f"Gerados {len(chunks)} chunks de ~{chunk_size} tokens.")


Gerados 993 chunks de ~250 tokens.


## Função Resposta Por similaridade

In [20]:
from numpy.linalg import norm
from scipy.spatial.distance import cosine

# pré‑calcula vetores médios dos chunks
chunk_vecs = []
for chunk in chunks:
    vecs = [w2v.wv[w] for w in chunk if w in w2v.wv]
    if vecs:
        chunk_vecs.append(np.mean(vecs, axis=0))
    else:
        chunk_vecs.append(np.zeros(w2v.vector_size))

def responde(pergunta, top_k=1):
    # pré‑processa pergunta
    doc = nlp(pergunta.lower())
    q_tokens = [tok.lemma_ for tok in doc
                if not tok.is_stop and not tok.is_punct and tok.lemma_ in w2v.wv]
    if not q_tokens:
        return "Não entendi — use palavras do livro."
    q_vec = np.mean([w2v.wv[w] for w in q_tokens], axis=0)

    # calcula similaridades
    sims = [1 - cosine(q_vec, cv) for cv in chunk_vecs]
    idxs = np.argsort(sims)[::-1][:top_k]

    respostas = []
    for i in idxs:
        texto = chunk_texts[i]
        respostas.append((sims[i], texto[:500] + "..."))  # mostra 500 chars
    return respostas


## Interface

In [21]:
import ipywidgets as widgets
from IPython.display import display, clear_output

input_pergunta = widgets.Text(
    placeholder="Digite sua pergunta sobre O Guarani",
    description="Pergunta:",
    layout=widgets.Layout(width="80%")
)
botao = widgets.Button(description="Responder")
saida = widgets.Output()

def on_click(b):
    with saida:
        clear_output()
        res = responde(input_pergunta.value, top_k=1)
        if isinstance(res, str):
            print(res)
        else:
            score, trecho = res[0]
            print(f"Similaridade: {score:.3f}\n\n{trecho}")

botao.on_click(on_click)

display(input_pergunta, botao, saida)


Text(value='', description='Pergunta:', layout=Layout(width='80%'), placeholder='Digite sua pergunta sobre O G…

Button(description='Responder', style=ButtonStyle())

Output()